# 🏥 Unified DICOM Burned-In Text Anonymizer Notebook
This notebook runs the **7-stage sequential DICOM anonymization pipeline** to remove PII/PHI from both the metadata headers and the image pixels (burned-in annotations) without losing diagnostic fidelity.

### ⚡ GPU Acceleration
OCR engines (PaddleOCR & EasyOCR) are computationally heavy. Running this notebook on a GPU environment (like Google Colab's free T4 GPU or Kaggle's GPU) will speed up processing by **5x - 10x**.

---

## 1. Environment Mount (For Google Colab Users Only)
If you are running this notebook in **Google Colab**, you can mount your Google Drive to easily upload raw DICOM files and download the anonymized results. If you are on **Kaggle**, you can skip this cell.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("[OK] Google Drive mounted successfully!")
except ImportError:
    print("[INFO] Not running in Google Colab (or Drive library not found). Skipping mount.")

## 2. Environment Setup (Install Dependencies)
Run the cell below to install all dependencies required for the pipeline. It installs `pydicom`, `easyocr`, `presidio-analyzer`, and the correct version of `paddlepaddle` based on whether a GPU is available.

In [ ]:
# Install core packages
!pip install -q pydicom easyocr presidio-analyzer opencv-python numpy matplotlib

# Check for GPU/CUDA presence and install matching PaddlePaddle engine
import torch
use_gpu = torch.cuda.is_available()
print(f"GPU (CUDA) Detected: {use_gpu}")

if use_gpu:
    print("Installing PaddlePaddle with GPU support...")
    !pip install -q paddlepaddle-gpu
else:
    print("Installing PaddlePaddle CPU-only...")
    !pip install -q paddlepaddle

# Install PaddleOCR
!pip install -q paddleocr
print("[OK] All dependencies installed successfully!")

## 3. Verify the Anonymizer Script
Make sure `dicom_anonymizer_pipeline.py` is in your current directory. Let's run a quick check to verify it is present:

In [ ]:
import os

script_name = "dicom_anonymizer_pipeline.py"
if os.path.exists(script_name):
    print(f"[OK] Found script: {os.path.abspath(script_name)}")
else:
    print(f"[ERROR] '{script_name}' not found in the current working directory!")
    print("Please upload or create 'dicom_anonymizer_pipeline.py' in the workspace.")

## 4. Path Configuration
Specify the paths to your raw DICOM file or directory of files, and where you want the anonymized output to go.

In [ ]:
# --- CONFIGURATION ---
# You can specify a single file path OR a folder path.
# Examples:
#   For Google Colab Drive: "/content/drive/MyDrive/raw_dicoms"
#   For Kaggle uploads: "/kaggle/input/your-dataset/scan.dcm"

INPUT_PATH = "./raw_datasamples_xray_chest"  # Change to your input file or directory
OUTPUT_PATH = "./anonymized_xray_chest"      # Change to your output directory
AUDIT_LOG_PATH = "./anonymized_xray_chest/audit_log.json"

# Set to True to use GPU (highly recommended), or False for CPU
USE_GPU = torch.cuda.is_available() 

# Create directories if they do not exist
import os
os.makedirs(OUTPUT_PATH, exist_ok=True)

## 5. Execute the Anonymizer Pipeline
You can run the anonymizer in two ways:

### Option A: Run via the Command Line Interface (CLI)
This runs the script directly as a process. It is highly recommended because it shows the log outputs in real-time as it processes each image.

In [ ]:
gpu_flag = "--gpu" if USE_GPU else "--no-gpu"
print(f"Starting anonymizer CLI execution (GPU={USE_GPU})...")

!python dicom_anonymizer_pipeline.py --input "{INPUT_PATH}" --output "{OUTPUT_PATH}" --audit "{AUDIT_LOG_PATH}" {gpu_flag}

### Option B: Run via Python API
Alternatively, you can import and invoke the `anonymize` function directly inside your python script.

In [ ]:
from dicom_anonymizer_pipeline import anonymize

print(f"Starting anonymizer API execution (GPU={USE_GPU})...")
success = anonymize(
    input_path=INPUT_PATH,
    output_path=OUTPUT_PATH,
    use_gpu=USE_GPU,
    audit_log_path=AUDIT_LOG_PATH
)

if success:
    print("\n[DONE] Anonymization finished successfully!")
else:
    print("\n[ERROR] Anonymization finished with errors. Check logs and audit_log.json.")

## 6. Verify and Visualize Results
Run the cell below to load the audit log and view a side-by-side visual comparison of the original DICOM image vs the redacted DICOM image.

In [ ]:
import pydicom
import numpy as np
import matplotlib.pyplot as plt
import os
import json

if os.path.exists(AUDIT_LOG_PATH):
    with open(AUDIT_LOG_PATH, 'r') as f:
        audit_data = json.load(f)
    print(f"Audit log loaded. Processed {len(audit_data)} files.")
    
    # Find the first successfully processed file to compare
    target_file = None
    for entry in audit_data:
        if entry["error"] is None:
            filename = entry["file"]
            raw_path = entry["input_path"]
            anon_path = entry["output_path"]
            if os.path.exists(raw_path) and os.path.exists(anon_path):
                target_file = (raw_path, anon_path, entry)
                break
                
    if target_file:
        raw_p, anon_p, entry = target_file
        print(f"\nComparing File: {entry['file']}")
        print(f"Redacted PHI Regions Count: {len(entry['redacted_regions'])}")
        for region in entry['redacted_regions']:
            print(f"  - Redacted text: '{region['text']}' @ Bounding Box: {region['bbox']}")
        
        # Load original and anonymized datasets
        ds_raw = pydicom.dcmread(raw_p, force=True)
        ds_anon = pydicom.dcmread(anon_p, force=True)
        
        # Extract pixel arrays
        if hasattr(ds_raw, 'pixel_array') and hasattr(ds_anon, 'pixel_array'):
            # Handle multi-frame by taking first frame
            img_raw = ds_raw.pixel_array[0] if ds_raw.pixel_array.ndim == 3 else ds_raw.pixel_array
            img_anon = ds_anon.pixel_array[0] if ds_anon.pixel_array.ndim == 3 else ds_anon.pixel_array
            
            # Check photometric inversion
            photo_raw = getattr(ds_raw, "PhotometricInterpretation", "MONOCHROME2")
            photo_anon = getattr(ds_anon, "PhotometricInterpretation", "MONOCHROME2")
            
            # Normalize helper
            def normalize_img(img, is_mono1):
                img_min, img_max = img.min(), img.max()
                if img_max > img_min:
                    norm = (img - img_min) / (img_max - img_min)
                else:
                    norm = img.copy().astype(np.float32)
                if is_mono1:
                    norm = 1.0 - norm  # invert MONOCHROME1 for consistent viewing
                return norm
            
            norm_raw = normalize_img(img_raw, photo_raw == "MONOCHROME1")
            norm_anon = normalize_img(img_anon, photo_anon == "MONOCHROME1")
            
            # Plot side-by-side
            fig, axes = plt.subplots(1, 2, figsize=(16, 10))
            
            axes[0].imshow(norm_raw, cmap='gray')
            axes[0].set_title(f"Original Scan (Header Name: {ds_raw.get('PatientName', 'N/A')})")
            axes[0].axis('off')
            
            axes[1].imshow(norm_anon, cmap='gray')
            axes[1].set_title(f"Anonymized Scan (Header Name: {ds_anon.get('PatientName', 'N/A')})")
            axes[1].axis('off')
            
            plt.tight_layout()
            plt.show()
        else:
            print("[WARN] DICOM dataset has no pixel array for visualization.")
    else:
        print("[WARN] No successful anonymized files were found for comparison.")
else:
    print("[WARN] Audit log json not found. Ensure you have run the anonymizer successfully first.")